# Primers Covid
 Copiar o descarga genoma covid

In [ ]:
##### ENVIRONMENT CREATION #####
module load blast/2.16.0_gcc-11.2.0 

##### JOB COMMANDS ####
makeblastdb -in Sars_cov.dna.fa -dbtype nucl -out sarscov2_db -title "SARS-CoV_genome"

4. Ahora realizamos Blast de la secuencia query (NRAMP5) contra la base de datos cacao_database

In [ ]:
##### ENVIRONMENT CREATION #####
module load blast/2.13.0_gcc-11.2.0

blastn -task blastn-short \
       -query primers.fa \
       -db sarscov2_db \
       -out primers_vs_sarscov2.txt \
       -outfmt "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore"

<div class="alert alert-block alert-info">

<b>Análisis de resultados de BLAST</b> 
- ¿Cómo interpretas esta tabla?
- ¿Qué tan específico es el primer?
- ¿En qué posición amplifica el fragmento?
- ¿Qué tan largo es el amplicón?
</div>

Ahora vamos a a extraer el fragmento que teóricamente debería amplificar este par de primers

In [ ]:
from Bio import SeqIO

# Cargamos el genoma
genoma = SeqIO.read("Sars_cov.dna.fa", "fasta")

# Mejores hits de BLAST output (extraídos manualmente)
forward_primer = "GACCCCAAAATCAGCGAAAT"       # example CDC N1 forward (put exact seq you used)
reverse_primer = "TCTGGTTACTGCCAGTTGAATCTG"   # example CDC N1 reverse (put exact seq you used)

# BLAST best-hit coordenadas
f_sstart, f_send = 109, 128   # forward: sstart, send: 28287, 28306
r_sstart, r_send = 511, 492   # reverse: sstart, send  (sstart>send indicates minus strand) 28358, 28335

# --- calcular las coordenas del producto (1-based) ---
# forward primer 5' on reference:
f_5 = min(f_sstart, f_send)   # 28287
# reverse primer 5' on reference is sstart when sstart>send (BLAST convention): 28358
r_5 = r_sstart if r_sstart >= r_send else r_send

amplicon_start = f_5
amplicon_end   = r_5
product_length = amplicon_end - amplicon_start + 1

print(f"Las coordenadas del amplicon son: {amplicon_start} - {amplicon_end}")
print(f"La longitud del produco es: {product_length} bp")


# --- extraer la secuencia del amplicon en FASTA ---

# Extraer el amplicon
amplicon_seq = genoma.seq[amplicon_start-1 : amplicon_end] 

print(amplicon_seq)

SeqIO.write(amplicon_record, "/home/lsalazarj/bicomp2025-2/primers/ampliconp1_covid.fa", "fasta")

## Verificación de especificidad con Primer-BLAST

Tomemos el par de primers del ensayo CDC N1 (gen N del SARS-CoV-2):

Forward (N1-F): GACCCCAAAATCAGCGAAAT

Reverse (N1-R): TCTGGTTACTGCCAGTTGAATCTG

Amplicón esperado: ~71 bp (posición 28,287–28,358 en la referencia NC_045512.2).

2. Abrir Primer-BLAST

Ir a 👉 NCBI Primer-BLAST

3. Configuración

En la sección Primer Parameters, introducir los primers:

Forward primer → GACCCCAAAATCAGCGAAAT

Reverse primer → TCTGGTTACTGCCAGTTGAATCTG

Database: seleccionar RefSeq representative genomes (virus).

Organism: Coronaviridae [11118] (para probar contra toda la familia de coronavirus).

Dejar los demás parámetros por defecto.